<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v3/modern-architectures/mixture-of-experts/mixture-of-experts.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("mixture-of-experts", ...)`


# Implement Mixture of Experts Layer

**Difficulty**: 🔴 Hard

**Companies**: Google, DeepMind, Mistral, Databricks, xAI

---

### Problem Statement

**Mixture of Experts (MoE)** is a technique for scaling model capacity without proportionally scaling compute. Instead of one large feed-forward network (FFN), MoE uses multiple smaller "expert" FFNs and a **gating/router network** that selects the **top-k** experts for each token. This means each token is only processed by a fraction of the total parameters.

A critical challenge in MoE is **load balancing**: without regularization, the router may collapse to always selecting the same expert(s), leaving others unused. An **auxiliary load balancing loss** encourages the router to distribute tokens evenly across experts.

Your task is to implement:
1. An `Expert` FFN module.
2. A `MoELayer` with top-k gating, expert dispatch, and load balancing loss.
3. A training loop that demonstrates the loss decreasing and experts receiving balanced load.

---

### Requirements

1. **`Expert(nn.Module)`** — A simple two-layer FFN: Linear → ReLU → Linear.
2. **`MoELayer(nn.Module)`** — Contains `self.experts` (an `nn.ModuleList` of `Expert`s) and `self.router` (an `nn.Linear(d_model, num_experts)`), and implements top-k gating with a weighted combination of expert outputs. `forward(x)` returns `(output, aux_loss)`.
3. **Load balancing loss** (Switch Transformer) — `aux_loss = num_experts * Σ_i f_i · P_i`, where `f_i` is the fraction of the `num_tokens × top_k` routing assignments that went to expert `i`, and `P_i` is the mean router probability (softmax over **all** experts) of expert `i`. It equals 1 under perfectly uniform routing and grows as routing concentrates on a few experts. `f_i` is a non-differentiable count; the gradient reaches the router through `P_i`. (Some codebases count `f_i` per token, so that `Σ f_i = top_k`; that only rescales the loss by `top_k`.)
4. **`MoEConfig`** — Dataclass with `num_experts`, `top_k`, `d_model`, `d_ff`, `aux_loss_weight`.

---

### Constraints

- ✅ Each token should be routed to exactly `top_k` experts.
- ✅ Each token's output is the gate-weighted sum of its `top_k` experts' outputs. The gates are renormalized over the selected experts: a softmax over the `top_k` selected router logits, which is identical to a softmax over all experts followed by renormalizing the top-k.
- ✅ All experts should receive some tokens after training (no expert collapse).
- ✅ The MoE has `num_experts`× the parameters of one expert FFN, but each token only runs through `top_k` experts, so per-token FLOPs scale with `top_k` (plus a small router), not with `num_experts`.
- ❌ Do **not** use any external MoE libraries.

> **Beyond this problem.** Production MoEs (GShard, Switch Transformer) also give each expert a fixed *capacity* of about `capacity_factor × num_tokens × top_k / num_experts` tokens and drop (or pass through on the residual) whatever overflows, so that expert batches have static shapes. This layer is *dropless*: every token gets all of its `top_k` experts. Note also that with `top_k = 1`, renormalizing would make every gate exactly 1 and cut the router off from the task-loss gradient — Switch therefore scales the expert output by the un-renormalized router probability.

---

<details>
  <summary>💡 Hint</summary>

  - Router: `gate_logits = self.router(x)` of shape `(batch*seq, num_experts)`. Use `torch.topk` to get top-k expert indices and values.
  - Normalize gate values: apply softmax only over the selected top-k values, not all experts.
  - Dispatch: loop over experts, find which tokens selected each expert, process them, and accumulate weighted outputs.
  - Load balance loss: `f_i` = fraction of routing assignments sent to expert i, `p_i` = mean router probability for expert i. Loss = `num_experts * sum(f_i * p_i)`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

In [ ]:
@dataclass
class MoEConfig:
    num_experts: int = 8
    top_k: int = 2
    d_model: int = 64
    d_ff: int = 128
    aux_loss_weight: float = 0.01

config = MoEConfig()
print(f"Config: {config}")

In [ ]:
class Expert(nn.Module):
    """A simple feed-forward expert network."""

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        # TODO: Create a two-layer FFN: Linear(d_model, d_ff) -> ReLU -> Linear(d_ff, d_model)
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Pass x through the FFN
        ...


class MoELayer(nn.Module):
    """Mixture of Experts layer with top-k gating and load balancing."""

    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config

        # TODO: Create self.experts: an nn.ModuleList of num_experts Expert modules

        # TODO: Create self.router: a Linear layer (d_model -> num_experts)

        ...

    def forward(self, x: torch.Tensor):
        """
        Forward pass with top-k expert routing.

        Args:
            x: (batch_size, seq_len, d_model)

        Returns:
            output: (batch_size, seq_len, d_model)
            aux_loss: scalar load balancing loss
        """
        batch_size, seq_len, d_model = x.shape
        # Flatten to (num_tokens, d_model) for routing
        x_flat = x.reshape(-1, d_model)
        num_tokens = x_flat.size(0)

        # TODO: Step 1 - Compute router logits: (num_tokens, num_experts)

        # TODO: Step 2 - Get top-k experts per token
        #       Use torch.topk on gate_logits to get top_k_values and top_k_indices

        # TODO: Step 3 - Gate weights: softmax over the k selected logits only

        # TODO: Step 4 - Dispatch tokens to experts and combine outputs
        #       Initialize output = zeros_like(x_flat)
        #       For each expert i:
        #         - Find which (token, slot) pairs selected expert i
        #         - Run only those tokens through expert i
        #         - Multiply by that slot's gate weight and ADD into those tokens' rows
        #           (a token visits k experts, so don't overwrite)

        # TODO: Step 5 - Compute auxiliary load balancing loss
        #       router_probs = softmax(gate_logits)  -- over ALL experts
        #       f_i = fraction of the num_tokens * top_k assignments sent to expert i
        #             (from top_k_indices; a count, no gradient)
        #       p_i = mean router probability for expert i (carries the gradient)
        #       aux_loss = num_experts * sum(f_i * p_i)   -> 1.0 when perfectly balanced

        # TODO: Reshape output back to (batch_size, seq_len, d_model)

        ...


# Quick test
torch.manual_seed(42)
moe = MoELayer(config)
x_test = torch.randn(4, 8, config.d_model)  # batch=4, seq=8
out, aux_loss = moe(x_test)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out.shape}")
print(f"Aux loss:     {aux_loss.item():.4f}  (1.0 = perfectly balanced)")

In [ ]:
# Training and validation
torch.manual_seed(42)

# Synthetic data: simple regression task
batch_size = 16
seq_len = 8
x_data = torch.randn(batch_size, seq_len, config.d_model)
# Target: an elementwise nonlinear transformation
y_data = torch.sin(x_data) * 0.5

moe = MoELayer(config)
optimizer = torch.optim.Adam(moe.parameters(), lr=1e-3)

losses = []
num_steps = 300

for step in range(num_steps):
    optimizer.zero_grad()
    output, aux_loss = moe(x_data)
    task_loss = F.mse_loss(output, y_data)
    total_loss = task_loss + config.aux_loss_weight * aux_loss
    total_loss.backward()
    optimizer.step()
    losses.append(total_loss.item())
    if step % 75 == 0:
        print(f"Step {step:3d} | Task loss: {task_loss.item():.4f} | Aux loss: {aux_loss.item():.4f} | Total: {total_loss.item():.4f}")

print(f"\nFinal total loss: {losses[-1]:.4f}")

# Validation 1: Loss should decrease
assert losses[-1] < losses[0], "Loss did not decrease!"
print("PASSED: Loss decreased.\n")

# Validation 2: Dispatch/combine must match a slow per-token reference
print("=== Per-Token Reference ===")
with torch.no_grad():
    x_check = torch.randn(2, 5, config.d_model)
    out_check, _ = moe(x_check)
    x_tokens = x_check.reshape(-1, config.d_model)
    top_vals, top_idx = torch.topk(moe.router(x_tokens), config.top_k, dim=-1)
    gates = F.softmax(top_vals, dim=-1)
    reference = torch.zeros_like(x_tokens)
    for t in range(x_tokens.size(0)):
        for g, e in zip(gates[t], top_idx[t].tolist()):
            reference[t] += g * moe.experts[e](x_tokens[t])
    max_diff = (out_check.reshape(-1, config.d_model) - reference).abs().max().item()
    assert max_diff < 1e-5, f"MoE output differs from the per-token reference (max diff {max_diff:.2e})"
    print(f"PASSED: matches sum of gate * expert(token) token by token (max diff {max_diff:.1e}).\n")

# Validation 3: Check load balance - all experts should receive tokens
print("=== Expert Load Balance ===")
with torch.no_grad():
    x_flat = x_data.view(-1, config.d_model)
    gate_logits = moe.router(x_flat)
    _, top_k_indices = torch.topk(gate_logits, config.top_k, dim=-1)
    expert_counts = torch.zeros(config.num_experts)
    for i in range(config.num_experts):
        expert_counts[i] = (top_k_indices == i).sum().item()
    total_assignments = top_k_indices.numel()
    for i in range(config.num_experts):
        pct = expert_counts[i] / total_assignments * 100
        print(f"  Expert {i}: {expert_counts[i]:5.0f} tokens ({pct:.1f}%)")
    assert (expert_counts > 0).all(), "Some experts received zero tokens (expert collapse)!"
    print("PASSED: All experts received tokens.\n")

# Validation 4: Parameter count comparison
print("=== Parameter Count Comparison ===")
moe_params = sum(p.numel() for p in moe.parameters())
dense_ffn = nn.Sequential(nn.Linear(config.d_model, config.d_ff), nn.ReLU(), nn.Linear(config.d_ff, config.d_model))
dense_params = sum(p.numel() for p in dense_ffn.parameters())
print(f"  MoE parameters:       {moe_params:,}")
print(f"  Dense FFN parameters: {dense_params:,}")
print(f"  Ratio: {moe_params / dense_params:.1f}x more parameters in MoE")
print(f"  But only top-{config.top_k}/{config.num_experts} experts active per token!")
print("PASSED\n")

print("All tests passed!")